# A setup for AI-assisted programming

This notebook describes how you can setup a system to use agents powered by large language models (LLMs) to assist you in programming tasks.

## Configuring access to the Imperial HX3 cluster

During these workshops we will be using LLMs which will be running on College infrastructure, namely the [HX3 cluster](https://icl-rcs-user-guide.readthedocs.io/en/latest/hpc/hx3/).
There are many reasons as to why one would choose to run one's own LLM, two important ones are:

* Data safety: your data is not shared with 3<sup>rd</sup> parties. This is very important in a research context, as you want to share your findings via a publication not via an LLM which will not attribute it to you.
* No need to pay a subscription to companies potentially engaged in unethical practices.

You should already be registered on the HX3 system. You now need to create a token (also called an API key) which will allow your agent to connect to the LLM running on HX3. There are 2 steps to do this, first generate a Client_ID/Client_secret pair, and then from those generate the actual token.

### Generate Client_ID/Client_secret (Access Key)

1. First connect to HX3 Run:ai platform: [https://runai.hx3.hpc.ic.ac.uk](https://runai.hx3.hpc.ic.ac.uk/).
2. You can ignore all the information on the screen and click the avatar on the top right corner of the screen, and choose the option *Settings*.
3. Choose the option *Access Keys*, and press the link `+ Access Key`.
4. Please note that when you press the button "Create", you need to copy the Client_secret pair, which will appear only once.
5. You can name this access key whatever you like. For example: ai-programming-workshop
6. On the next screen you will see the Client_ID/Client_secret pair that was generated (you will need to click on the 'eye' icon to see the client secret. Copy the Client_ID and Client_secret values to the code cell below. After you copy the value you can press `Done` and close the Run:ai webpage.

In [ ]:
runai_client_id=""
runai_client_secret=""

### Generating tokens/API key

The code cell below uses your Client_ID/Client_secret pair defined above to connect to HX3 and generate a token.

In [ ]:
import urllib.request
import urllib.parse
import json

data = {
    "grantType": "client_credentials",
    "clientId": runai_client_id,
    "clientSecret": runai_client_secret
}

req = urllib.request.Request(
    "https://runai.hx3.hpc.ic.ac.uk/api/v1/token",
    data=json.dumps(data).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(req) as response:
    response_data = json.loads(response.read().decode("utf-8"))
    
token = response_data["accessToken"]
print(token)

Tokens generated by HX3 expire after about one hour. We will use the token generated here during configuration of the agent below, but instead of repeatedly generating tokens ourselves we will configure our agent to call a script with code above and generate a new token when needed.

Edit the script *HX3_tokens.py* on the subfolder *goose_config/* and add your Client_ID/Client_secret pair. Note that your Client_ID/Client_secret pair is sensitive information, equivalent to login/password pair, so you want to keep the script private and safe.

## Installing software

For the purpose of these workshops we will install 2 pieces of software:

**Goose**
: This is our agent *harness*, which passes on our queries to the LLM and relay its output. Importantly, the agent does not simply display the text output by the LLM, but it is also able to perform actions, such as creating new files, or be asked to read the content of files or websites, or use other tools.

**Jupyter AI**
: This is a Jupyter extension providing the interface of Jupyter Lab with AI agents.

### Installing and configuring Goose CLI

We now want to install the command line interface (CLI) version of Goose. We will describe the installation procedure here, but you can consult the [Goose installation instructions](https://goose-docs.ai/docs/getting-started/installation) reference from the website.

#### Installation on Windows

Open the PowerShell and run the following command

    Invoke-WebRequest -Uri "https://raw.githubusercontent.com/aaif-goose/goose/main/download_cli.ps1" -OutFile "download_cli.ps1";
    
This will download the installation script *download_cli.ps1*. We now run the script by typing in the PowerShell

    .\download_cli_ps1
    
This will download and install Goose CLI files and initiate the configuration process. Once you are asked the first question of the configuration process you can interrupt the configuration by pressing the key combination `Ctrl+C`. 
Then move to the configuration step below.

#### Installation on macOS or Linux

Open a terminal and run the following commnad

    curl -fsSL https://github.com/aaif-goose/goose/releases/download/stable/download_cli.sh | CONFIGURE=false bash

Continue with the configuration step.

#### Configuration

We have prepared some configuration files which you will find on the *goose_config/* directory accompanying this notebook. The following 2 files need to be edited:

* goose_config/custom_providers/custom_hx3_devstral.json
* goose_config/custom_providers/custom_hx3_qwen.json

On each of those files, look for the line containing `"args:"` which follows the line `"command": "python",`. You need to replace the string *"_pointer_to_HX3_tokens.py"*, with the final full path to that file. The output of the cell below makes an educated guess of what the path is. Copy it into the relevant files.

In [ ]:
import os

try:
    config_path = os.path.abspath(os.path.join(os.environ["APPDATA"],"Block","goose","config"))

except KeyError:
    config_path = os.path.abspath(os.path.join(os.environ["HOME"],".config","goose"))

if os.path.exists(config_path):
    print(os.path.join(config_path,"HX3_tokens.py"))
else:
    print("HELP! I'm lost.")

After editing these files, we want to copy the full content of the *goose_config/* to the actual Goose configuration directory. Running the cell below will attempt to do this for you, provided the cell above did not give an error.

In [ ]:
import shutil

shutil.copytree("goose_config",config_path,dirs_exist_ok=True)

This completes the configuration. To test if the agent is correctly configured you can start a session on the command line by typing in the PowerShell or terminal

    goose session
    
If you see an error about the Jupyter extension, this is safe to ignore for now until we install Jupyter AI below. Any other errors are unexpected and need to be fixed.

If you see no other errors, try querying the model!

### Installing Jupyter AI

We assume that you have a conda-forge or anaconda environment installed. If you don't contact a demonstrator.

Open the Miniforge/Anaconda prompt or a terminal, and type

    conda install jupyter-ai
    
That should be it. You can start jupyter lab by typing in the terminal

    jupyter lab
    
In the jupyter lab interface you should now have the possibility to open a chat window. You should see Goose as the agent at the bottom of the chat. Ask it a question!